# 2D Slice MS-SWD + MDS Diversity Analysis

This notebook converts each 3D digital rock volume into 2D slices along the three orthogonal axes (`Z`, `Y`, and `X`). The original 2D MS-SWD-MDS workflow is then applied to the resulting slice set: slices are grouped, each group aggregates multi-scale 2D Laplacian patch distributions, group-to-group average SWD distances are used to build a distance matrix, and MDS maps the matrix to 2D for visualization.


In [ ]:
import gc
import glob
import os
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.lines import Line2D
from sklearn.manifold import MDS

# ==========================================
# 1. Data, model groups, and 2D MS-SWD parameters
# ==========================================
DIMENSION = (128, 128, 128)  # interpreted as (Z, Y, X)
DATA_TYPE = np.int8
PORE_VALUE = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FOLDERS = {
    "TI": r"../../data/real_binary",
    "DCGAN": r"../../data/baselines/dcgan",
    "StyleGAN": r"../../data/baselines/stylegan",
    "STGAN": r"../../data/baselines/stgan",
    "ours": r"../../data/generated_binary",
}

COLORS = {
    "TI": "#e41a1c",
    "DCGAN": "#4daf4a",
    "StyleGAN": "#377eb8",
    "STGAN": "#222222",
    "ours": "#984ea3",
}

# Use all three orthogonal slicing directions. Increase SLICE_STRIDE for a faster debug run.
SLICE_AXES = ("z", "y", "x")
SLICE_STRIDE = 1
MAX_FILES_PER_MODEL = None      # Set to 1 or 2 for debugging; use None for the full run.
MAX_SLICES_PER_MODEL = None     # Set to e.g. 400 for debugging; use None for all slices.

# Paper-style 2D MS-SWD settings.
GROUP_SIZE = 40
PYRAMID_LEVELS = 3
PATCH_SIZE = 5
NUM_PATCHES_PER_GROUP = 2048
NUM_PROJECTIONS = 256
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Current device: {DEVICE}")


In [ ]:
# ==========================================
# 2. Load binary raw rocks and extract 2D slices from Z/Y/X axes
# ==========================================
def list_raw_files(folder, max_files=None):
    files = sorted(glob.glob(os.path.join(folder, "*.raw")))
    if max_files is not None:
        files = files[:max_files]
    return files


def load_binary_rock(filepath):
    raw = np.fromfile(filepath, dtype=DATA_TYPE)
    expected = int(np.prod(DIMENSION))
    if raw.size != expected:
        raise ValueError(
            f"{filepath} has {raw.size} voxels, expected {expected}. "
            f"Check DIMENSION or DATA_TYPE."
        )
    volume = raw.reshape(DIMENSION)
    return (volume == PORE_VALUE).astype(np.uint8)


def extract_axis_slices(volume, axes=SLICE_AXES, stride=SLICE_STRIDE):
    slices = []
    d, h, w = volume.shape

    if "z" in axes:
        for z in range(0, d, stride):
            slices.append(volume[z, :, :].copy())
    if "y" in axes:
        for y in range(0, h, stride):
            slices.append(volume[:, y, :].copy())
    if "x" in axes:
        for x in range(0, w, stride):
            slices.append(volume[:, :, x].copy())

    return slices


def load_all_slices():
    slice_images = []
    slice_labels = []
    slice_sources = []

    for method, folder in FOLDERS.items():
        files = list_raw_files(folder, MAX_FILES_PER_MODEL)
        method_slices = []
        method_sources = []
        print(f"\n[{method}] found {len(files)} raw files: {folder}")

        for file_idx, file_path in enumerate(files, start=1):
            volume = load_binary_rock(file_path)
            file_slices = extract_axis_slices(volume)
            method_slices.extend(file_slices)
            method_sources.extend([(file_path, idx) for idx in range(len(file_slices))])
            print(f"  sliced volume {file_idx}/{len(files)} -> {len(file_slices)} slices")
            del volume, file_slices
            gc.collect()

        if MAX_SLICES_PER_MODEL is not None and len(method_slices) > MAX_SLICES_PER_MODEL:
            selected = rng.choice(len(method_slices), size=MAX_SLICES_PER_MODEL, replace=False)
            method_slices = [method_slices[i] for i in selected]
            method_sources = [method_sources[i] for i in selected]

        slice_images.extend(method_slices)
        slice_labels.extend([method] * len(method_slices))
        slice_sources.extend(method_sources)
        print(f"  total slices used for {method}: {len(method_slices)}")

    if len(slice_images) < 2:
        raise RuntimeError("Need at least two valid 2D slices to run MDS.")
    return slice_images, slice_labels, slice_sources


slice_images, slice_labels, slice_sources = load_all_slices()
print(f"\nTotal 2D slices: {len(slice_images)}")


In [ ]:
# ==========================================
# 3. Build 2D Laplacian pyramids and sample 2D patches
# ==========================================
def laplacian_pyramid_2d(image_tensor, levels=PYRAMID_LEVELS):
    """Return 2D Laplacian pyramid levels for one image tensor, shape (1,1,H,W)."""
    pyramid = []
    current = image_tensor
    for _ in range(levels - 1):
        low = F.avg_pool2d(current, kernel_size=2, stride=2)
        up = F.interpolate(low, size=current.shape[-2:], mode="bilinear", align_corners=False)
        pyramid.append((current - up).squeeze(0).squeeze(0))
        current = low
    pyramid.append(current.squeeze(0).squeeze(0))
    return pyramid


def sample_patches_from_level(level_tensor, num_patches):
    h, w = level_tensor.shape
    if min(h, w) < PATCH_SIZE:
        raise ValueError(f"Pyramid level shape {tuple(level_tensor.shape)} is too small for PATCH_SIZE={PATCH_SIZE}.")

    ys = torch.randint(0, h - PATCH_SIZE + 1, (num_patches,), device=DEVICE)
    xs = torch.randint(0, w - PATCH_SIZE + 1, (num_patches,), device=DEVICE)

    patches = torch.empty((num_patches, PATCH_SIZE ** 2), device=DEVICE)
    for i in range(num_patches):
        patches[i] = level_tensor[
            ys[i] : ys[i] + PATCH_SIZE,
            xs[i] : xs[i] + PATCH_SIZE,
        ].flatten()

    # Normalize each patch by its own mean and standard deviation, as in the paper.
    mean = patches.mean(dim=1, keepdim=True)
    std = patches.std(dim=1, keepdim=True) + 1e-8
    return ((patches - mean) / std).cpu()


def extract_pyramid_patches_from_slice(image, num_patches_per_level):
    tensor = torch.from_numpy(image.astype(np.float32, copy=False)).to(DEVICE).unsqueeze(0).unsqueeze(0)
    levels = laplacian_pyramid_2d(tensor, levels=PYRAMID_LEVELS)
    patches = [sample_patches_from_level(level, num_patches_per_level) for level in levels]
    del tensor, levels
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return patches


In [ ]:
# ==========================================
# 4. Group slices and aggregate patch distributions for each group
# ==========================================
def make_groups(items, group_size, shuffle=True):
    items = list(items)
    if shuffle:
        rng.shuffle(items)
    groups = [items[i : i + group_size] for i in range(0, len(items), group_size)]
    return [g for g in groups if len(g) >= 2]


def build_group_features(group_slices):
    patches_per_slice = int(np.ceil(NUM_PATCHES_PER_GROUP / len(group_slices)))
    per_level = [[] for _ in range(PYRAMID_LEVELS)]

    for image in group_slices:
        image_patches = extract_pyramid_patches_from_slice(image, patches_per_slice)
        for level_idx, patches in enumerate(image_patches):
            per_level[level_idx].append(patches)

    group_features = []
    for level_idx in range(PYRAMID_LEVELS):
        patches = torch.cat(per_level[level_idx], dim=0)
        if len(patches) > NUM_PATCHES_PER_GROUP:
            idx = torch.randperm(len(patches))[:NUM_PATCHES_PER_GROUP]
            patches = patches[idx]
        group_features.append(patches.contiguous())
    return group_features


group_features = []
group_labels = []

for method in FOLDERS.keys():
    method_slices = [img for img, label in zip(slice_images, slice_labels) if label == method]
    groups = make_groups(method_slices, GROUP_SIZE)
    if not groups:
        print(f"Warning: no valid groups for {method}")
        continue

    print(f"\n[{method}] {len(method_slices)} slices -> {len(groups)} groups, group_size={GROUP_SIZE}")
    for idx, slices_in_group in enumerate(groups, start=1):
        group_features.append(build_group_features(slices_in_group))
        group_labels.append(method)
        print(f"  built group {idx}/{len(groups)}", end="\r")
        gc.collect()
    print()

if len(group_features) < 2:
    raise RuntimeError("Need at least two valid groups to run MDS.")

print(f"\nTotal MDS points/groups: {len(group_features)}")


In [ ]:
# ==========================================
# 5. Average MS-SWD distance matrix between 2D slice groups
# ==========================================
_projection_cache = {}


def get_projection(dim, level_idx):
    key = (dim, level_idx)
    if key not in _projection_cache:
        torch.manual_seed(RANDOM_SEED + level_idx * 1009 + dim)
        proj = torch.randn(dim, NUM_PROJECTIONS, device=DEVICE)
        proj = proj / (torch.norm(proj, dim=0, keepdim=True) + 1e-8)
        _projection_cache[key] = proj
    return _projection_cache[key]


def swd_between_patch_sets(patches_a, patches_b, level_idx):
    a = patches_a.to(DEVICE)
    b = patches_b.to(DEVICE)
    n = min(len(a), len(b))
    if len(a) != n:
        a = a[torch.randperm(len(a), device=DEVICE)[:n]]
    if len(b) != n:
        b = b[torch.randperm(len(b), device=DEVICE)[:n]]

    proj = get_projection(a.shape[1], level_idx)
    pa = torch.sort(a @ proj, dim=0)[0]
    pb = torch.sort(b @ proj, dim=0)[0]
    dist = torch.mean(torch.abs(pa - pb)).item()
    del a, b, pa, pb
    return dist


def ms_swd_between_groups(group_a, group_b):
    distances = []
    for level_idx, (patches_a, patches_b) in enumerate(zip(group_a, group_b)):
        distances.append(swd_between_patch_sets(patches_a, patches_b, level_idx))
    return float(np.mean(distances))


def build_group_distance_matrix(group_features):
    n = len(group_features)
    matrix = np.zeros((n, n), dtype=np.float64)
    total = n * (n - 1) // 2
    print(f"Start calculating {n} x {n} group MS-SWD distance matrix, total pairs: {total}")
    for current, (i, j) in enumerate(combinations(range(n), 2), start=1):
        matrix[i, j] = matrix[j, i] = ms_swd_between_groups(group_features[i], group_features[j])
        print(f"  progress: {current}/{total}", end="\r")
    print()
    return matrix


dist_matrix = build_group_distance_matrix(group_features)
np.save("slice2d_ms_swd_group_distance_matrix.npy", dist_matrix)
print("Saved distance matrix to slice2d_ms_swd_group_distance_matrix.npy")


In [ ]:
# ==========================================
# 6. MDS + bounded local KDE density plot
# ==========================================
def fit_mds(dist_matrix):
    try:
        mds = MDS(n_components=2, dissimilarity="precomputed", random_state=RANDOM_SEED, normalized_stress="auto")
    except TypeError:
        mds = MDS(n_components=2, dissimilarity="precomputed", random_state=RANDOM_SEED)
    return mds.fit_transform(dist_matrix)


def pairwise_distances_2d(pts):
    if len(pts) < 2:
        return np.array([])
    diffs = pts[:, None, :] - pts[None, :, :]
    d = np.sqrt(np.sum(diffs * diffs, axis=-1))
    return d[np.triu_indices(len(pts), k=1)]


def bounded_local_kde_contour(ax, pts, color, linestyle, levels=5):
    """Draw bounded KDE-like contours for small MDS group counts."""
    pts = np.asarray(pts, dtype=float)
    if len(pts) < 2:
        return False

    pair_d = pairwise_distances_2d(pts)
    positive = pair_d[pair_d > 0]
    if positive.size:
        bandwidth = np.median(positive) * 0.4
    else:
        bandwidth = max(np.ptp(pts[:, 0]), np.ptp(pts[:, 1]), 1e-4) * 0.35
    bandwidth = max(float(bandwidth), 1e-5)

    pad = bandwidth * 2.4
    x_min, x_max = pts[:, 0].min() - pad, pts[:, 0].max() + pad
    y_min, y_max = pts[:, 1].min() - pad, pts[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 180), np.linspace(y_min, y_max, 180))

    zz = np.zeros_like(xx, dtype=float)
    min_dist = np.full_like(xx, np.inf, dtype=float)
    for point in pts:
        dist2 = (xx - point[0]) ** 2 + (yy - point[1]) ** 2
        zz += np.exp(-0.5 * dist2 / (bandwidth ** 2))
        min_dist = np.minimum(min_dist, np.sqrt(dist2))
    zz /= len(pts)
    zz = np.where(min_dist <= bandwidth * 4, zz, np.nan)

    finite = zz[np.isfinite(zz) & (zz > 0)]
    if finite.size < 10:
        return False

    contour_levels = np.quantile(finite, np.linspace(0.58, 0.92, levels))
    contour_levels = np.unique(contour_levels)
    if len(contour_levels) < 2:
        return False

    ax.contour(
        xx,
        yy,
        zz,
        levels=contour_levels,
        colors=[color],
        linestyles=linestyle,
        linewidths=1,
        zorder=3,
    )
    return True


def set_square_axes(ax, coords_2d):
    x_min, x_max = coords_2d[:, 0].min(), coords_2d[:, 0].max()
    y_min, y_max = coords_2d[:, 1].min(), coords_2d[:, 1].max()
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    half_span = max(x_max - x_min, y_max - y_min) / 2
    half_span += max(half_span * 0.22, 1e-6)
    ax.set_xlim(x_center - half_span, x_center + half_span)
    ax.set_ylim(-0.15,0.15)
    ax.set_aspect("equal", adjustable="box")
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)


def plot_density_mds(coords_2d, labels):
    labels = np.asarray(labels)
    fig, ax = plt.subplots(figsize=(5, 5), dpi=600)

    line_styles = {
        "TI": ("#ec3336", "-"),
        "DCGAN": ("#bb38c7", "--"),
        "STGAN": ("#8633D3", "--"),
        "StyleGAN": ("#328EDA", "--"),
        "ours": ("#3DD816", "--"),
    }

    handles = []
    for method in FOLDERS.keys():
        pts = coords_2d[labels == method]
        if len(pts) == 0:
            continue
        color, linestyle = line_styles.get(method, (COLORS.get(method, "#666666"), "-"))
        ok = bounded_local_kde_contour(ax, pts, color=color, linestyle=linestyle)
        if ok:
            handles.append(Line2D([0], [0], color=color, linestyle=linestyle, lw=1.8, label=method))

    ax.set_title("Densityplot", fontsize=12, pad=4)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", labelsize=8)
    ax.legend(handles=handles, loc="upper right", frameon=True, fontsize=8)
    set_square_axes(ax, coords_2d)

    fig.subplots_adjust(left=0.14, right=0.96, bottom=0.12, top=0.96)
    plt.savefig("slice2d_ms_swd_mds_densityplot.png", dpi=600)
    plt.savefig("slice2d_ms_swd_mds_densityplot.pdf")
    plt.show()


coords_2d = fit_mds(dist_matrix)
plot_density_mds(coords_2d, group_labels)
print("Saved figures: slice2d_ms_swd_mds_densityplot.png / slice2d_ms_swd_mds_densityplot.pdf")


In [ ]:
# ==========================================
# 7. MDS scatter plot
# ==========================================
def plot_scatter_mds(coords_2d, labels):
    labels = np.asarray(labels)
    fig, ax = plt.subplots(figsize=(5, 5), dpi=600)

    markers = {
        "TI": "o",
        "DCGAN": "s",
        "STGAN": "^",
        "StyleGAN": "D",
        "ours": "P",
    }

    for method in FOLDERS.keys():
        pts = coords_2d[labels == method]
        if len(pts) == 0:
            continue
        ax.scatter(
            pts[:, 0],
            pts[:, 1],
            s=24,
            marker=markers.get(method, "o"),
            color=COLORS.get(method, "#666666"),
            edgecolors="white",
            linewidths=0.45,
            alpha=0.88,
            label=method,
            zorder=3,
        )

    ax.set_title("Scatterplot", fontsize=12, pad=4)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", labelsize=8)
    ax.legend(loc="upper right", frameon=True, fontsize=9)
    set_square_axes(ax, coords_2d)

    fig.subplots_adjust(left=0.14, right=0.96, bottom=0.12, top=0.96)
    plt.savefig("slice2d_ms_swd_mds_scatterplot.png", dpi=600)
    plt.savefig("slice2d_ms_swd_mds_scatterplot.pdf")
    plt.show()


if "coords_2d" not in globals():
    coords_2d = fit_mds(dist_matrix)

plot_scatter_mds(coords_2d, group_labels)
print("Saved figures: slice2d_ms_swd_mds_scatterplot.png / slice2d_ms_swd_mds_scatterplot.pdf")


In [ ]:
# ==========================================
# 8. t-SNE scatter plot based on the MS-SWD distance matrix
# ==========================================
from sklearn.manifold import TSNE


def fit_tsne(distance_matrix, random_state=RANDOM_SEED):
    n = distance_matrix.shape[0]
    perplexity = min(30, max(2, (n - 1) // 3))

    try:
        tsne = TSNE(
            n_components=2,
            metric="precomputed",
            init="random",
            perplexity=perplexity,
            random_state=random_state,
            learning_rate="auto",
        )
    except TypeError:
        tsne = TSNE(
            n_components=2,
            metric="precomputed",
            init="random",
            perplexity=perplexity,
            random_state=random_state,
            learning_rate=200,
        )

    return tsne.fit_transform(distance_matrix)


def plot_tsne_scatter(coords_2d, labels):
    labels = np.asarray(labels)
    fig, ax = plt.subplots(figsize=(5, 5), dpi=600)

    markers = {
        "TI": "o",
        "DCGAN": "s",
        "STGAN": "^",
        "StyleGAN": "D",
        "ours": "P",
    }

    for method in FOLDERS.keys():
        pts = coords_2d[labels == method]
        if len(pts) == 0:
            continue
        ax.scatter(
            pts[:, 0],
            pts[:, 1],
            s=26,
            marker=markers.get(method, "o"),
            color=COLORS.get(method, "#666666"),
            edgecolors="white",
            linewidths=0.45,
            alpha=0.9,
            label=method,
            zorder=3,
        )

    ax.set_title("t-SNE", fontsize=12, pad=4)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", labelsize=8)
    ax.legend(loc="upper right", frameon=True, fontsize=9)
    set_square_axes(ax, coords_2d)

    fig.subplots_adjust(left=0.14, right=0.96, bottom=0.12, top=0.96)
    plt.savefig("slice2d_ms_swd_tsne_scatterplot.png", dpi=600)
    plt.savefig("slice2d_ms_swd_tsne_scatterplot.pdf")
    plt.show()


tsne_coords_2d = fit_tsne(dist_matrix)
plot_tsne_scatter(tsne_coords_2d, group_labels)
print("Saved figures: slice2d_ms_swd_tsne_scatterplot.png / slice2d_ms_swd_tsne_scatterplot.pdf")
